<br/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="left"/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="right"/>
<div align="center">
<h2>Bootcamp Data Science — Módulo 2</h2><br/>
<h1>Semana 5 · Miércoles — Encoding de Variables Categóricas</h1>
<h3>OrdinalEncoder · OneHotEncoder · LabelEncoder · get_dummies</h3>
<br/>
    <b>Instructor:</b> Jesús Ortiz · jesus.jeduardo7@gmail.com<br/><br/>
    <b>SkillNest · b2b-sonda-data-science</b>
</div>
<br/>

## 🎯 Objetivos de hoy

Al final de la clase vas a poder:

1. Entender **por qué** los modelos de ML necesitan números.
2. Distinguir entre variable **ordinal** y **nominal** y elegir el método correcto.
3. Dominar **4 métodos de encoding** y sus diferencias:
   - `OrdinalEncoder` (sklearn)
   - `LabelEncoder` (sklearn — solo para target)
   - `OneHotEncoder` (sklearn — para pipelines)
   - `pd.get_dummies()` (pandas — atajo rápido)
4. Manejar **casos límite**: categorías nuevas en test, multicolinealidad, alta cardinalidad.
5. Resolver **5 ejercicios prácticos** con decisiones reales.

# 1. ¿Por qué importa codificar?

Los algoritmos de ML trabajan con **operaciones matemáticas** (sumas, multiplicaciones, distancias). Si tu columna dice `'Premium'` o `'Santiago'`, el algoritmo **no puede operar con texto** — necesitamos traducirlas a números.

Pero hay **una decisión crítica** antes de codificar: ¿la variable tiene un **orden natural** o no?

| Tipo | ¿Tiene orden? | Ejemplos | Método correcto |
|---|---|---|---|
| **Ordinal** | ✅ Sí | Talla (S<M<L), Educación, Calificación 1★→5★ | `OrdinalEncoder` |
| **Nominal** | ❌ No | Color, País, Marca, Género musical | `OneHotEncoder` / `get_dummies` |

### ⚠️ Por qué importa la diferencia

Si codificas `colores = ['rojo', 'azul', 'verde']` como `[0, 1, 2]`, el modelo va a interpretar que:

> *"verde (2) > azul (1) > rojo (0)"*

Eso no tiene sentido y **degrada el modelo**. El modelo aprende relaciones falsas. Por eso necesitamos métodos diferentes según el tipo de variable.

# 2. OrdinalEncoder — para variables CON orden

## ¿Qué hace exactamente?

`OrdinalEncoder` asigna un número entero a cada categoría **respetando un orden que TÚ defines**.

Si no le indicas el orden, sklearn los ordena alfabéticamente — que casi siempre es lo que NO querés. Por eso **siempre** hay que pasar el parámetro `categories=[[...]]`.

## Parámetros importantes

| Parámetro | Para qué sirve |
|---|---|
| `categories` | Lista de listas con el orden EXPLÍCITO de cada columna |
| `handle_unknown` | Qué hacer si en test aparece una categoría no vista en train (`'error'` por defecto, `'use_encoded_value'` la reemplaza) |
| `unknown_value` | Valor para categorías desconocidas (ej. `-1`) |

### ✅ Ejemplo bien hecho: variable con orden real

In [3]:
df = pd.DataFrame({
    'calidad': ['bad', 'good', 'good', 'average', 'bad', 'average', 'good', 'average']
})

df

,calidad
0,bad
1,good
2,good
3,average
4,bad
5,average
6,good
7,average


In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder

# Variable con orden REAL: calidad de un producto
df = pd.DataFrame({
    'calidad': ['bad', 'good', 'good', 'average', 'bad', 'average', 'good', 'average','jesus']
})

# Le indicamos el orden EXPLÍCITAMENTE — esto es OBLIGATORIO
encoder = OrdinalEncoder(categories=[['bad', 'average', 'good']], handle_unknown='use_encoded_value', unknown_value=np.nan)
encoder.fit(df[['calidad']])

df['calidad_cod'] = encoder.transform(df[['calidad']])

print(df)
print(f'\nMapping aprendido: bad=0, average=1, good=2')
print(f'Categorías guardadas: {encoder.categories_}')

   calidad  calidad_cod
0      bad          0.0
1     good          2.0
2     good          2.0
3  average          1.0
4      bad          0.0
5  average          1.0
6     good          2.0
7  average          1.0
8    jesus          NaN

Mapping aprendido: bad=0, average=1, good=2
Categorías guardadas: [array(['bad', 'average', 'good'], dtype=object)]


### 🚨 Ejemplo MAL hecho: aplicar OrdinalEncoder a algo sin orden

Esta es la trampa más común. Veamos el problema con colores:

In [4]:
df_colores = pd.DataFrame({
    'color': ['rojo', 'azul', 'azul', 'verde', 'rojo', 'verde']
})

# Aplicamos OrdinalEncoder SIN especificar orden — sklearn lo hace alfabético
encoder_malo = OrdinalEncoder()
df_colores['cod'] = encoder_malo.fit_transform(df_colores[['color']])

print(df_colores)
print(f'\nMapping resultante: {encoder_malo.categories_[0]}')
print('   azul=0, rojo=1, verde=2')
print()
print('🚨 PROBLEMA:')
print('   El modelo va a pensar que verde > rojo > azul')
print('   Eso no es cierto, son solo nombres de colores')
print('   → Para esto debemos usar OneHotEncoder (siguiente sección)')

   color  cod
0   rojo  1.0
1   azul  0.0
2   azul  0.0
3  verde  2.0
4   rojo  1.0
5  verde  2.0

Mapping resultante: ['azul' 'rojo' 'verde']
   azul=0, rojo=1, verde=2

🚨 PROBLEMA:
   El modelo va a pensar que verde > rojo > azul
   Eso no es cierto, son solo nombres de colores
   → Para esto debemos usar OneHotEncoder (siguiente sección)


### ⚠️ Caso límite: una categoría NUEVA en test

¿Qué pasa si en `X_train` hay `['bad', 'average', 'good']` y en `X_test` aparece `'excellent'`? Por defecto, sklearn **lanza un error**. Hay que decirle qué hacer.

In [9]:
# Train con 3 categorías
train = pd.DataFrame({'calidad': ['bad', 'average', 'good']})
test  = pd.DataFrame({'calidad': ['good', 'excellent', 'bad']})  # 'excellent' es NUEVA

# Opción robusta: manejar categorías desconocidas
encoder = OrdinalEncoder(
    categories=[['bad', 'average', 'good']],
    handle_unknown='use_encoded_value',
    unknown_value=-1   # 'excellent' será codificado como -1
)
encoder.fit(train[['calidad']])

test['cod'] = encoder.transform(test[['calidad']])
print(test)
print('\n👉 "excellent" queda como -1, que el modelo puede interpretar como "desconocido".')

     calidad  cod
0       good  2.0
1  excellent -1.0
2        bad  0.0

👉 "excellent" queda como -1, que el modelo puede interpretar como "desconocido".


# 3. LabelEncoder — el primo de OrdinalEncoder (solo para target)

Esta es una confusión muy común. **`LabelEncoder` se ve igual que `OrdinalEncoder`**, pero hay una diferencia clave:

| | `OrdinalEncoder` | `LabelEncoder` |
|---|---|---|
| ¿Dónde se usa? | En **features (X)** | Solo en **target (y)** |
| ¿Acepta 2D? | ✅ Sí (DataFrames con varias columnas) | ❌ No, solo 1D (una columna) |
| ¿Permite definir orden? | ✅ Sí (`categories=`) | ❌ No, siempre alfabético |

**Regla práctica:** si vas a codificar la columna que vas a PREDECIR (variable objetivo, target), usá `LabelEncoder`. Si vas a codificar features (X), usá `OrdinalEncoder` o `OneHotEncoder`.

In [11]:
from sklearn.preprocessing import LabelEncoder

# Caso típico: clasificación binaria — target es 'enfermo' o 'sano'
y = pd.Series(['sano', 'enfermo', 'sano', 'sano', 'enfermo', 'maso'])

le = LabelEncoder()
y_cod = le.fit_transform(y)

print(f'Original:    {y.values}')
print(f'Codificado:  {y_cod}')
print(f'Mapping:     {dict(zip(le.classes_, range(len(le.classes_))))}')
print('\n👉 LabelEncoder hace alfabético: enfermo=0, sano=1')

Original:    ['sano' 'enfermo' 'sano' 'sano' 'enfermo' 'maso']
Codificado:  [2 0 2 2 0 1]
Mapping:     {'enfermo': 0, 'maso': 1, 'sano': 2}

👉 LabelEncoder hace alfabético: enfermo=0, sano=1


# 4. OneHotEncoder — para variables SIN orden

## ¿Qué hace exactamente?

En vez de asignar un número, **crea una columna binaria por cada categoría**. Cada fila tiene un `1` en su categoría y `0` en las demás.

Para `colores = ['rojo', 'azul', 'verde']`:

| color | rojo | azul | verde |
|---|---|---|---|
| rojo | 1 | 0 | 0 |
| azul | 0 | 1 | 0 |
| verde | 0 | 0 | 1 |

Así, **todas las categorías están a la misma distancia** entre sí — no hay orden artificial.

## Parámetros importantes

| Parámetro | Para qué sirve |
|---|---|
| `sparse_output` | Si es `True` devuelve matriz dispersa (eficiente en memoria). Para ver el resultado mejor `False` |
| `drop` | `'first'` o `'if_binary'`: elimina una columna para evitar **multicolinealidad** |
| `handle_unknown` | `'error'` o `'ignore'`: cómo manejar categorías nuevas en test |

In [13]:
df = pd.DataFrame({'color': ['rojo', 'azul', 'verde', 'rojo', 'azul']})
df

,color
0,rojo
1,azul
2,verde
3,rojo
4,azul


In [17]:
from sklearn.preprocessing import OneHotEncoder

df = pd.DataFrame({'color': ['rojo', 'azul', 'verde', 'rojo', 'azul']})

# sparse_output=False para verlo como matriz normal
ohe = OneHotEncoder(sparse_output=False)
matriz = ohe.fit_transform(df[['color']])

# Convertimos a DataFrame para mostrar bonito
df_codificado = pd.DataFrame(matriz, columns=ohe.get_feature_names_out(['color']))
print('Resultado:')
print(df_codificado)
print(f'\nCategorías aprendidas: {ohe.categories_}')

Resultado:
   color_azul  color_rojo  color_verde
0         0.0         1.0          0.0
1         1.0         0.0          0.0
2         0.0         0.0          1.0
3         0.0         1.0          0.0
4         1.0         0.0          0.0

Categorías aprendidas: [array(['azul', 'rojo', 'verde'], dtype=object)]


### 🚧 El problema de la multicolinealidad — `drop='first'`

Si una variable tiene 3 categorías y creamos 3 columnas binarias, **una de ellas es redundante**:

Si sabemos que `azul=0` y `verde=0`, automáticamente sabemos que `rojo=1`. Las 3 columnas están **perfectamente correlacionadas** — esto se llama **dummy trap** o **multicolinealidad**.

**Para modelos lineales** (regresión lineal, logística), esto puede degradar la interpretación. La solución es **eliminar UNA columna**:

In [18]:
# Con drop='first' eliminamos la primera columna (azul)
ohe_drop = OneHotEncoder(sparse_output=False, drop='first')
matriz_drop = ohe_drop.fit_transform(df[['color']])

df_drop = pd.DataFrame(matriz_drop, columns=ohe_drop.get_feature_names_out(['color']))
print('Con drop="first" (azul es la "referencia"):')
print(df_drop)
print()
print('👉 Si rojo=0 y verde=0 → es azul. La info está ahí, pero sin redundancia.')
print('   Esto es lo recomendado para modelos lineales como LinearRegression.')

Con drop="first" (azul es la "referencia"):
   color_rojo  color_verde
0         1.0          0.0
1         0.0          0.0
2         0.0          1.0
3         1.0          0.0
4         0.0          0.0

👉 Si rojo=0 y verde=0 → es azul. La info está ahí, pero sin redundancia.
   Esto es lo recomendado para modelos lineales como LinearRegression.


# 5. pd.get_dummies() — el atajo de pandas

Pandas tiene una función nativa que hace **lo mismo que OneHotEncoder**, pero en una línea:

```python
pd.get_dummies(df, columns=['mi_categorica'])
```

## OneHotEncoder vs get_dummies — ¿cuál usar?

| Aspecto | `OneHotEncoder` | `pd.get_dummies()` |
|---|---|---|
| Sintaxis | Verbosa (fit/transform) | 1 línea |
| Funciona en pipelines | ✅ Sí | ❌ No |
| Recuerda categorías | ✅ Sí (clave para test) | ❌ No |
| Manejo de NaN | Configurable | Crea columna `'_nan'` por defecto |
| Output | Numpy array / sparse | DataFrame con nombres |
| `drop_first` | `drop='first'` | `drop_first=True` |

### Regla práctica

- 🔍 **Exploración / análisis rápido** → `pd.get_dummies()`
- 🏭 **Producción / pipelines con train+test** → `OneHotEncoder`

In [20]:
df = pd.DataFrame({
    'cliente':  [1, 2, 3, 4, 5],
    'plan':     ['Básico', 'Premium', 'Básico', 'Premium', 'Plus'],
    'ciudad':   ['Santiago', 'Valparaíso', 'Santiago', 'Concepción', 'Santiago']
})
df

,cliente,plan,ciudad
0,1,Básico,Santiago
1,2,Premium,Valparaíso
2,3,Básico,Santiago
3,4,Premium,Concepción
4,5,Plus,Santiago


In [21]:
df_dummies

,cliente,plan_Básico,plan_Plus,plan_Premium,ciudad_Concepción,ciudad_Santiago,ciudad_Valparaíso
0,1,1,0,0,0,1,0
1,2,0,0,1,0,0,1
2,3,1,0,0,0,1,0
3,4,0,0,1,1,0,0
4,5,0,1,0,0,1,0


In [ ]:
df = pd.DataFrame({
    'cliente':  [1, 2, 3, 4, 5],
    'plan':     ['Básico', 'Premium', 'Básico', 'Premium', 'Plus'],
    'ciudad':   ['Santiago', 'Valparaíso', 'Santiago', 'Concepción', 'Santiago']
})

# Codifica TODAS las categóricas con una sola línea
df_dummies = pd.get_dummies(df, columns=['plan', 'ciudad'], dtype=int)
df_dummies

,cliente,plan_Básico,plan_Plus,plan_Premium,ciudad_Concepción,ciudad_Santiago,ciudad_Valparaíso
0,1,1,0,0,0,1,0
1,2,0,0,1,0,0,1
2,3,1,0,0,0,1,0
3,4,0,0,1,1,0,0
4,5,0,1,0,0,1,0


In [ ]:
# Con drop_first para evitar multicolinealidad
df_dummies_drop = pd.get_dummies(df, columns=['plan', 'ciudad'], drop_first=True, dtype=int)
df_dummies_drop

# 6. ⚠️ Casos especiales y trampas

## Alta cardinalidad — cuando hay MUCHAS categorías

Si tu columna tiene **país** (200 valores) o **código postal** (5000+ valores), one-hot crea 200 o 5000 columnas — esto es **explosión dimensional** y mata el modelo.

**Estrategias:**
1. **Agrupar categorías raras** en `'Otros'` (las que aparecen < 1% del tiempo).
2. **Target encoding** (avanzado — reemplazar cada categoría por el promedio del target).
3. **Frequency encoding** (reemplazar cada categoría por su frecuencia).

Hoy nos enfocamos solo en la estrategia 1 — la más simple.

In [23]:
df = pd.DataFrame({
    'pais': ['Chile']*50 + ['Perú']*30 + ['Brasil']*15 + 
            ['Argentina']*3 + ['Uruguay']*1 + ['Paraguay']*1
})
df

,pais
0,Chile
1,Chile
2,Chile
3,Chile
4,Chile
...,...
95,Argentina
96,Argentina
97,Argentina
98,Uruguay


In [22]:
# Ejemplo: dataset con muchos países
df = pd.DataFrame({
    'pais': ['Chile']*50 + ['Perú']*30 + ['Brasil']*15 + 
            ['Argentina']*3 + ['Uruguay']*1 + ['Paraguay']*1
})

print('Frecuencia de países:')
print(df['pais'].value_counts())

# Agrupar los que aparecen < 5 veces en 'Otros'
umbral = 5
frecuentes = df['pais'].value_counts()
raros = frecuentes[frecuentes < umbral].index
df['pais_agrupado'] = df['pais'].replace(raros, 'Otros')

print('\nDespués de agrupar:')
print(df['pais_agrupado'].value_counts())
print('\n👉 Pasamos de 6 categorías a 4. One-hot creará menos columnas.')

Frecuencia de países:
pais
Chile        50
Perú         30
Brasil       15
Argentina     3
Uruguay       1
Paraguay      1
Name: count, dtype: int64

Después de agrupar:
pais_agrupado
Chile     50
Perú      30
Brasil    15
Otros      5
Name: count, dtype: int64

👉 Pasamos de 6 categorías a 4. One-hot creará menos columnas.


# 7. Ejemplo completo con dataset real

In [24]:
# Cargamos housing — esta vez con la categórica
df = pd.read_csv('data/housing.csv').dropna()

print(f'Shape: {df.shape}')
print(f'\nValores de ocean_proximity:')
print(df['ocean_proximity'].value_counts())

Shape: (20433, 10)

Valores de ocean_proximity:
ocean_proximity
<1H OCEAN     9034
INLAND        6496
NEAR OCEAN    2628
NEAR BAY      2270
ISLAND           5
Name: count, dtype: int64


In [26]:
df_cod

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,0,0,1,0
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,0,0,1,0
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,0,0,1,0
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,0,0,1,0
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,1,0,0,0
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,1,0,0,0
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,1,0,0,0
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,1,0,0,0


In [25]:
# FLUJO CORRECTO: encoding + estandarización + train/test + modelo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

# 1. Encoding de la categórica
df_cod = pd.get_dummies(df, columns=['ocean_proximity'], drop_first=True, dtype=int)

# 2. Features y target
X = df_cod.drop(columns=['median_house_value'])
y = df_cod['median_house_value']

# 3. Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Estandarización
scaler = StandardScaler()
X_train_esc = scaler.fit_transform(X_train)
X_test_esc  = scaler.transform(X_test)

# 5. Modelo
modelo = LinearRegression().fit(X_train_esc, y_train)
print(f'R²: {modelo.score(X_test_esc, y_test):.4f}')
print(f'Features usadas: {X.shape[1]} (incluye {sum("ocean_proximity" in c for c in X.columns)} de la categórica)')

R²: 0.6488
Features usadas: 12 (incluye 4 de la categórica)


---
# 🏋️ Ejercicios prácticos — con decisiones reales

**Datasets disponibles:**
- `data/housing.csv` — `ocean_proximity`
- `data/life_expectancy.csv` — `Status` (Developed / Developing)
- `seaborn.load_dataset('tips')` — `sex`, `smoker`, `day`, `time`
- `seaborn.load_dataset('penguins')` — `species`, `island`, `sex`

## Ejercicio 1 — Diagnóstico: ¿qué método usar?

Para cada caso, decide **qué método de encoding usarías** y justifica. Hay decisiones reales que tomar.

| # | Variable | Valores | Tu decisión |
|---|---|---|---|
| 1 | `talla_polera` | S, M, L, XL, XXL | ? |
| 2 | `color_auto` | rojo, blanco, negro, gris | ? |
| 3 | `pais_origen` | 180 países distintos | ? |
| 4 | `nivel_satisfaccion` | 1★, 2★, 3★, 4★, 5★ | ? |
| 5 | `tipo_pago` | tarjeta, efectivo | ? |
| 6 | `target_enfermedad` | sí, no | ? |
| 7 | `categoria_producto` | 'Electronics', 'Books', 'Toys', 'Food' | ? |

**Para cada uno, di:**
- El método (`OrdinalEncoder`, `OneHotEncoder`/`get_dummies`, `LabelEncoder`)
- Si requiere alguna preparación previa (ej: agrupar categorías raras)

<details><summary>💡 Solución</summary>

| # | Variable | Método | Justificación |
|---|---|---|---|
| 1 | `talla_polera` | **OrdinalEncoder** con `categories=[['S','M','L','XL','XXL']]` | Hay orden lógico |
| 2 | `color_auto` | **OneHotEncoder / get_dummies** | No hay orden, son 4 categorías → OK |
| 3 | `pais_origen` | **OneHotEncoder pero PRIMERO agrupar raros** | 180 = alta cardinalidad. Agrupar los menos frecuentes en 'Otros' |
| 4 | `nivel_satisfaccion` | **OrdinalEncoder** | 1★ < 5★ → orden claro |
| 5 | `tipo_pago` | **OneHotEncoder con `drop='first'`** | Binaria → 1 columna basta |
| 6 | `target_enfermedad` | **LabelEncoder** | Es el target, no feature |
| 7 | `categoria_producto` | **OneHotEncoder / get_dummies** | Sin orden entre categorías |

</details>

## Ejercicio 2 — Encoding mixto con dataset `tips`

El dataset `tips` tiene **5 columnas categóricas** de distintos tipos. Tu tarea es decidir y aplicar:

1. Cargar el dataset:
   ```python
   import seaborn as sns
   tips = sns.load_dataset('tips')
   ```
2. Explorar `tips.dtypes` y los valores únicos de cada columna categórica.
3. Aplicar el método **correcto a cada una**:
   - `sex`: ?
   - `smoker`: ?
   - `day`: ¿es ordinal o nominal? (los días tienen orden si son fines de semana vs semana)
   - `time`: ?
4. Construir un DataFrame final solo con columnas numéricas + las codificadas.
5. Reportar el shape final.

**Pista:** `day` los podemos tratar como **ordinales** si decimos que `Thur < Fri < Sat < Sun`.

In [ ]:
# Tu código aquí 👇



<details><summary>💡 Solución</summary>

```python
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder

tips = sns.load_dataset('tips')
print('Tipos:'); print(tips.dtypes)

# 1. sex, smoker, time → binarias sin orden → OneHotEncoder con drop_first
# 2. day → puede tratarse como ordinal (Thur < Fri < Sat < Sun)

# Day como ordinal
encoder = OrdinalEncoder(categories=[['Thur', 'Fri', 'Sat', 'Sun']])
tips['day_ord'] = encoder.fit_transform(tips[['day']])

# Resto con get_dummies (drop_first para binarias)
tips_cod = pd.get_dummies(tips.drop(columns=['day']),
                          columns=['sex', 'smoker', 'time'],
                          drop_first=True, dtype=int)

print(f'\nShape final: {tips_cod.shape}')
tips_cod.head()
```
</details>

## Ejercicio 3 — La trampa de las categorías nuevas

Imagina que entrenas un modelo con cierto conjunto de categorías y, en producción, llega una categoría que **nunca viste antes**. ¿Qué hace tu modelo?

**Tarea:**

1. Crear el `train` con esta data:
   ```python
   train = pd.DataFrame({'ciudad': ['Santiago', 'Valparaíso', 'Concepción']*4})
   ```
2. Crear el `test` con esta data (con `'Antofagasta'` que NO está en train):
   ```python
   test = pd.DataFrame({'ciudad': ['Santiago', 'Antofagasta', 'Valparaíso']})
   ```
3. Intentar codificar `test` con `OneHotEncoder` ajustado con `train` SIN configurar `handle_unknown`. ¿Qué error sale?
4. Repetir con `handle_unknown='ignore'`. ¿Qué pasa con `Antofagasta`?
5. Explicar en una línea: ¿qué hace `handle_unknown='ignore'`?

In [ ]:
# Tu código aquí 👇



<details><summary>💡 Solución</summary>

```python
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

train = pd.DataFrame({'ciudad': ['Santiago', 'Valparaíso', 'Concepción']*4})
test  = pd.DataFrame({'ciudad': ['Santiago', 'Antofagasta', 'Valparaíso']})

# 1. Sin handle_unknown → ERROR
ohe = OneHotEncoder(sparse_output=False)
ohe.fit(train[['ciudad']])
try:
    ohe.transform(test[['ciudad']])
except ValueError as e:
    print(f'❌ Error: {e}')

# 2. Con handle_unknown='ignore'
ohe2 = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe2.fit(train[['ciudad']])
matriz = ohe2.transform(test[['ciudad']])
print('\nResultado con handle_unknown="ignore":')
print(pd.DataFrame(matriz, columns=ohe2.get_feature_names_out(['ciudad'])))

# 👉 'Antofagasta' queda con todos ceros → el modelo lo trata como "otro".
#    Esto es CLAVE para producción: una categoría nueva no rompe tu pipeline.
```
</details>

## Ejercicio 4 — Comparar `drop_first=True` vs `drop_first=False`

Vamos a verificar si la **multicolinealidad** afecta o no el R² en regresión lineal.

**Tarea:**

1. Cargar `data/housing.csv` con `.dropna()`.
2. Entrenar 2 modelos comparando:
   - Modelo A: `get_dummies(... drop_first=True)` → quitar primera categoría de `ocean_proximity`
   - Modelo B: `get_dummies(... drop_first=False)` → mantener todas
3. Usar el mismo `random_state=42` y mismo flujo (estandarizar + LinearRegression).
4. Comparar el R² de ambos.
5. ¿Cambió mucho? ¿Por qué?

In [ ]:
# Tu código aquí 👇



<details><summary>💡 Solución</summary>

```python
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

df = pd.read_csv('data/housing.csv').dropna()
y = df['median_house_value']

def evaluar(drop_first, etiqueta):
    df_cod = pd.get_dummies(df, columns=['ocean_proximity'], drop_first=drop_first, dtype=int)
    X = df_cod.drop(columns=['median_house_value'])
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_test  = sc.transform(X_test)
    r2 = LinearRegression().fit(X_train, y_train).score(X_test, y_test)
    print(f'{etiqueta:30s} → R²={r2:.4f}, features={X.shape[1]}')

evaluar(True,  'Con drop_first=True')
evaluar(False, 'Sin drop_first (todas)')

# 👉 Los R² son prácticamente IDÉNTICOS.
#    sklearn maneja la redundancia internamente.
#    Pero drop_first=True usa MENOS features → modelo más simple y limpio.
```
</details>

## Ejercicio 5 — Desafío integrador con `penguins`

Vamos a predecir el **peso de un pingüino** (`body_mass_g`) usando todo lo aprendido.

**Tarea:**

1. Cargar:
   ```python
   penguins = sns.load_dataset('penguins').dropna()
   ```
2. Variables:
   - Target: `body_mass_g`
   - Features: el resto
3. Identificar qué columnas son categóricas y aplicar encoding adecuado:
   - `species`: nominal → one-hot
   - `island`: nominal → one-hot
   - `sex`: binaria → one-hot con drop_first o un solo número
4. Train/test 80/20, `random_state=42`.
5. Estandarizar correctamente.
6. Entrenar `LinearRegression` y reportar **R²**, **MAE** y **RMSE**.
7. **Bonus:** ¿qué pasa con el R² si NO codificas las categóricas y solo usas las numéricas?

In [27]:
# Tu código aquí 👇
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

penguins = sns.load_dataset('penguins').dropna()

# Codificación de categóricas
df_cod = pd.get_dummies(penguins, columns=['species', 'island', 'sex'],
                        drop_first=True, dtype=int)

X = df_cod.drop(columns=['body_mass_g'])
y = df_cod['body_mass_g']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
sc = StandardScaler()
X_train_esc = sc.fit_transform(X_train)
X_test_esc  = sc.transform(X_test)

modelo = LinearRegression().fit(X_train_esc, y_train)
y_pred = modelo.predict(X_test_esc)

print(f'R²:   {modelo.score(X_test_esc, y_test):.4f}')
print(f'MAE:  {mean_absolute_error(y_test, y_pred):.2f}g')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}g')

# BONUS: sin categóricas
X_sin = penguins.select_dtypes(include='number').drop(columns=['body_mass_g'])
Xtr, Xte, ytr, yte = train_test_split(X_sin, y, test_size=0.2, random_state=42)
sc2 = StandardScaler()
Xtr = sc2.fit_transform(Xtr); Xte = sc2.transform(Xte)
print(f'\nSin codificar categóricas: R²={LinearRegression().fit(Xtr, ytr).score(Xte, yte):.4f}')


R²:   0.8962
MAE:  196.21g
RMSE: 255.75g

Sin codificar categóricas: R²=0.7981


<details><summary>💡 Solución</summary>

```python
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

penguins = sns.load_dataset('penguins').dropna()

# Codificación de categóricas
df_cod = pd.get_dummies(penguins, columns=['species', 'island', 'sex'],
                        drop_first=True, dtype=int)

X = df_cod.drop(columns=['body_mass_g'])
y = df_cod['body_mass_g']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
sc = StandardScaler()
X_train_esc = sc.fit_transform(X_train)
X_test_esc  = sc.transform(X_test)

modelo = LinearRegression().fit(X_train_esc, y_train)
y_pred = modelo.predict(X_test_esc)

print(f'R²:   {modelo.score(X_test_esc, y_test):.4f}')
print(f'MAE:  {mean_absolute_error(y_test, y_pred):.2f}g')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}g')

# BONUS: sin categóricas
X_sin = penguins.select_dtypes(include='number').drop(columns=['body_mass_g'])
Xtr, Xte, ytr, yte = train_test_split(X_sin, y, test_size=0.2, random_state=42)
sc2 = StandardScaler()
Xtr = sc2.fit_transform(Xtr); Xte = sc2.transform(Xte)
print(f'\nSin codificar categóricas: R²={LinearRegression().fit(Xtr, ytr).score(Xte, yte):.4f}')
# 👉 Suele bajar ~0.10. species y sex aportan muchísima señal — los machos pesan más, ciertas especies son más grandes.
```
</details>

---
## 📌 Cierre del día

Hoy aprendimos a fondo:

- ✅ Por qué los modelos necesitan números (no texto)
- ✅ Cuándo usar **OrdinalEncoder** (variables con orden lógico)
- ✅ Cuándo usar **LabelEncoder** (solo para target)
- ✅ Cuándo usar **OneHotEncoder** (variables sin orden)
- ✅ El atajo **`pd.get_dummies()`** y cuándo conviene usarlo
- ✅ La trampa de la **multicolinealidad** y el rol de `drop_first`
- ✅ Cómo manejar **categorías nuevas en test** con `handle_unknown='ignore'`
- ✅ El problema de la **alta cardinalidad** (agrupar categorías raras)

### 🔜 Próximas clases

- **Viernes 22** (después del feriado): Árboles de decisión para regresión
- **Semana 6:** Pipelines, ColumnTransformer, SimpleImputer, KNN, Random Forest

Nos vemos 🚀